# Colab runner

Launcher only — no method code lives here. See `model.py`.

**Connect first:** `Select Kernel` → `Colab` → `New Colab Server` → pick **GPU**.
Then run these cells top to bottom.

Everything below executes on the Colab VM, not on your laptop.


## 1. Confirm we actually got a GPU

In [1]:
!nvidia-smi
import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

Sat Sep 19 17:32:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Drive

(Or use the command palette: `Colab: Mount Google Drive to Server...`)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Clone the repo onto the VM

`/content` is wiped when the runtime dies, so this runs every fresh session.

In [6]:
%cd /content
!git clone https://github.com/Dev-Joyson/CNN-based-GAN-face-detection.git 2>/dev/null || (cd CNN-based-GAN-face-detection && git pull)
%cd /content/CNN-based-GAN-face-detection
!pip install -q pyyaml

/content
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.83 KiB | 937.00 KiB/s, done.
From https://github.com/Dev-Joyson/CNN-based-GAN-face-detection
   c67dbae..136314f  main       -> origin/main
Updating c67dbae..136314f
Fast-forward
 audit_dataset.py             |  6 ++---
 model.py                     | 13 ++++++++++-
 notebooks/colab_runner.ipynb | 52 +++++++++++++++++++++++++++++++++++++++++---
 3 files changed, 64 insertions(+), 7 deletions(-)
/content/CNN-based-GAN-face-detection


## 4. Sanity check: do the guard tests pass on this machine?

~3 seconds, no dataset needed.

In [4]:
!pytest tests -q

............                                                             [100%]
12 passed in 10.99s


## 5. Audit the dataset for shortcuts

No GPU needed. Scores each trivial file property as an AUC — if any reaches ~0.9,
the model can hit that score without looking at a face.

In [ ]:
!python audit_dataset.py --config configs/test16_full.yaml -n 800

2026-09-19 17:46:00.462707: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-19 17:46:00.531963: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
real: 70000 files   fake: 25000 files
sampling 800 of each...

2026-09-19 17:46:07.686138: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1789839967.687699    5628 gpu_device.cc:2020] Created device /job:localhost/replica

## 6. Train the headline model (full image, no mask)

Epoch 1 is slow — it reads every image off Drive and writes the cache to
local disk. Later epochs read the cache and are much faster. Don't kill it.

Outputs go to Drive (`out_dir` in the config), so they survive a disconnect.


In [ ]:
!python train.py --config configs/test16_full.yaml


## 7. Controls: face-only and background-only

The panel's question was whether the model reads the background. These two
runs answer it. They use a different dataset/size to test16, so they build
their own cache.


In [ ]:
!python train.py --config configs/test13_face.yaml
!python train.py --config configs/test14_background.yaml


## 8. Evaluate: confusion matrix, ROC, Grad-CAM

Writes `confusion_matrix.png`, `roc.png`, `gradcam.png` and `eval.json` into the
run folder in Drive. Uses the same masked pipeline as training, so the numbers
match what the model actually saw.

Grad-CAM is the shortcut check: under `face_only` the heat should sit on the face.


In [ ]:
!python evaluate.py --config configs/test16_full.yaml
!python evaluate.py --config configs/test13_face.yaml
!python evaluate.py --config configs/test14_background.yaml


## 9. Watch training

**Live:** TensorBoard reads `<run>/tb/` and refreshes itself every 30 s.
Pointing it at the experiments root shows every run on one chart.
If the panel does not render inside VS Code, open this same notebook at
colab.research.google.com — it embeds there.

**Snapshot:** the cell after reads `history.csv` from Drive; re-run it any time.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/Research/experiments"


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv("/content/drive/MyDrive/Research/experiments/test16_full/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
h[["accuracy", "val_accuracy"]].plot(ax=ax[0], title="accuracy")
h[["auc", "val_auc"]].plot(ax=ax[1], title="AUC")
plt.tight_layout(); plt.show()